In [ ]:
import pandas as pd
import os
import csv  # for quoting constants

OUTPUT_FOLDER = "Combined_Outputs_Columns_Removed"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

file_aws_table = "Cleaned_With_Columns/Cleaned_AWS_CSVs/padded_cdi_ca_1956_wk_prov_dbs_Part1_tables.csv"
file_aws_confidence = "Cleaned_With_Columns/Cleaned_AWS_CSVs/padded_cdi_ca_1956_wk_prov_dbs_Part1_confidences.csv"
file_manual_table = "Cleaned_With_Columns/Cleaned_Manual_CSVs/padded_cdi_ca_1956_wk_prov_dbs_Part1.csv"


def is_empty_or_dash(value):
    # Return True if the cell should NOT be used for numeric comparison.
    s = str(value).strip()
    # Empty, whitespace, dash, dots
    if s == "" or s == "-" or s.replace(".", "") == "":
        return True
    return False


def get_value(df, i, j):
    return df.iat[i, j]


def normalize_commas_spaces(value):
    return str(value).replace(",", "").replace(" ", "")


def extract_numeric_string(value):
    s = str(value)
    numeric = "".join(ch for ch in s if ch.isdigit() or ch in "+-.")
    return numeric


def only_numbers_equal(df1, df3, i, j):
    v1 = extract_numeric_string(get_value(df1, i, j))
    v2 = extract_numeric_string(get_value(df3, i, j))
    return v1 == v2


def cells_equal(df1, df3, i, j):
    # Raw equality on whatever is in the dataframes (now strings from CSV)
    return get_value(df1, i, j) == get_value(df3, i, j)


def cells_equal_no_commas_spaces(df1, df3, i, j):
    v1 = normalize_commas_spaces(get_value(df1, i, j))
    v2 = normalize_commas_spaces(get_value(df3, i, j))
    return v1 == v2


def cells_equal_shifted(df1, df3, i, j, n_cols):
    """Shifted equality + equality ignoring commas/spaces."""
    v1 = str(get_value(df1, i, j))
    v2 = str(get_value(df3, i, j))
    # Remove commas and spaces for normalized comparisons
    v1_norm = v1.replace(",", "").replace(" ", "")
    v2_norm = v2.replace(",", "").replace(" ", "")
    shifted_equal = False
    if j + 1 < n_cols:
        shifted_raw = str(get_value(df1, i, j + 1))
        shifted_norm = shifted_raw.replace(",", "").replace(" ", "")
        shifted_equal = (v2_norm == shifted_norm)

    # OR rule: shifted OR normalized equals
    return shifted_equal or (v1_norm == v2_norm)


def cells_numbers_only_equal(df1, df3, i, j):
    v1 = get_value(df1, i, j)
    v2 = get_value(df3, i, j)
    # Skip if either is empty/dash/dots
    if is_empty_or_dash(v1) or is_empty_or_dash(v2):
        return ""
    n1 = extract_numeric_string(v1)
    n2 = extract_numeric_string(v2)
    if not n1 or not n2:
        return ""
    return n1 == n2


def combine_in_csvs(file_aws_table, file_aws_confidence, file_manual_table):

    # === IMPORTANT CSV CHANGES HERE ===
    # Read ALL values as raw strings, no NA conversion.
    df1 = pd.read_csv(
        file_aws_table,
        header=None,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
    )
    df2 = pd.read_csv(
        file_aws_confidence,
        header=None,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
    )
    df3 = pd.read_csv(
        file_manual_table,
        header=None,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
    )

    # Match the smallest shape among the three (prevents index errors)
    n_rows = min(df1.shape[0], df2.shape[0], df3.shape[0])
    n_cols = min(df1.shape[1], df2.shape[1], df3.shape[1])

    rows = []
    for i in range(n_rows):
        for j in range(n_cols):
            eq = cells_equal(df1, df3, i, j)
            eq_nc = cells_equal_no_commas_spaces(df1, df3, i, j)
            eq_shift = cells_equal_shifted(df1, df3, i, j, n_cols)
            num_only = cells_numbers_only_equal(df1, df3, i, j)
            rows.append({
                "file_name": os.path.basename(file_aws_table),
                "row_index": i,
                "col_index": j,
                # These come straight from df1/df3 which are raw strings from CSV
                "AWS_Value": get_value(df1, i, j),
                "Manual_Value": get_value(df3, i, j),
                # Both values are compared exactly as Python sees them (strings).
                "Equal": eq,
                "Equal_No_Commas_No_Spaces": eq_nc,
                # Shifted AND no commas/spaces equality.
                "Equal_Shifted": eq_shift,
                # doesn't consider empty/dash/dots cells, only compares numeric parts.
                # AND the shifted equality.
                "Only_Numbers_Equal_Or_Shifted": bool(num_only) or eq_shift,
                "Any_Equality": (
                    eq or
                    eq_nc or
                    eq_shift or
                    bool(num_only)
                ),
                "Confidence_Value": df2.iat[i, j],
            })

    combined = pd.DataFrame(rows)

    # Build output filename from the input CSV name
    base_name = os.path.basename(file_manual_table).replace(".csv", "")
    output_path = os.path.join(OUTPUT_FOLDER, f"{base_name}_combined.csv")

    combined.to_csv(
        output_path,
        index=False,
        quoting=csv.QUOTE_MINIMAL,
    )
    print(f"Saved: {output_path}")

    return combined


# Batch over all 52 parts
for i in range(1, 53):
    file_aws_table = f"Cleaned_With_Columns/Cleaned_AWS_CSVs/padded_cdi_ca_1956_wk_prov_dbs_Part{i}_tables.csv"
    file_aws_confidence = f"Cleaned_With_Columns/Cleaned_AWS_CSVs/padded_cdi_ca_1956_wk_prov_dbs_Part{i}_confidences.csv"
    file_manual_table = f"Cleaned_With_Columns/Cleaned_Manual_CSVs/padded_cdi_ca_1956_wk_prov_dbs_Part{i}.csv"
    combine_in_csvs(file_aws_table, file_aws_confidence, file_manual_table)


NameError: name 'file_aws_table' is not defined

In [2]:
import pandas as pd
import os
import glob

def combine_all_csvs(input_folder, output_file):
    # Find all .csv files in the folder
    pattern = os.path.join(input_folder, "*.csv")
    csv_files = glob.glob(pattern)

    if not csv_files:
        raise ValueError("No CSV files found in the folder.")

    dfs = []
    for f in csv_files:
        df = pd.read_csv(f)
        dfs.append(df)

    # Combine vertically
    combined_df = pd.concat(dfs, ignore_index=True)

    # Save output
    combined_df.to_csv(output_file, index=False)
    print(f"Saved combined file: {output_file}")
    print(f"Rows combined: {combined_df.shape[0]}")
    return combined_df

combine_all_csvs("Combined_Outputs_Columns_Removed", "C:\\Users\\edeno\\Downloads\\eden\\Textract Pipeline Experiments\\Levels_of_Accuracy.csv")

Saved combined file: C:\Users\edeno\Downloads\eden\Textract Pipeline Experiments\Levels_of_Accuracy.csv
Rows combined: 56756


,file_name,row_index,col_index,AWS_Value,Manual_Value,Equal,Equal_No_Commas_No_Spaces,Equal_Shifted,Only_Numbers_Equal_Or_Shifted,Any_Equality,Confidence_Value
0,padded_cdi_ca_1956_wk_prov_dbs_Part10_tables.csv,0,0,1.0,1,True,False,False,False,True,85.11
1,padded_cdi_ca_1956_wk_prov_dbs_Part10_tables.csv,0,1,Chickenpox,Chickenpox,True,True,True,True,True,91.99
2,padded_cdi_ca_1956_wk_prov_dbs_Part10_tables.csv,0,2,"1,111",1111,False,True,True,True,True,86.38
3,padded_cdi_ca_1956_wk_prov_dbs_Part10_tables.csv,0,3,"1,234",1234,False,True,True,True,True,82.32
4,padded_cdi_ca_1956_wk_prov_dbs_Part10_tables.csv,0,4,"1,188",1188,False,True,True,True,True,88.67
...,...,...,...,...,...,...,...,...,...,...,...
56751,padded_cdi_ca_1956_wk_prov_dbs_Part9_tables.csv,27,34,11,11,True,True,True,True,True,38.06
56752,padded_cdi_ca_1956_wk_prov_dbs_Part9_tables.csv,27,35,39,39,True,True,True,True,True,68.7
56753,padded_cdi_ca_1956_wk_prov_dbs_Part9_tables.csv,27,36,14,14,True,True,True,True,True,38.57
56754,padded_cdi_ca_1956_wk_prov_dbs_Part9_tables.csv,27,37,19,19,True,True,True,True,True,68.07


In [4]:
# python code that opens a csv and prints the average of a column specified (average of true/false)
import csv

def average_boolean_column(csv_file, column_name):
    total = 0
    count = 0

    with open(csv_file, newline='') as f:
        reader = csv.DictReader(f)

        for row in reader:
            value = row[column_name].strip().lower()

            if value in ("true", "1"):
                total += 1
                count += 1
            elif value in ("false", "0"):
                count += 1
            else:
                # skip empty or invalid values
                continue

    if count == 0:
        print("No valid values found.")
    else:
        avg = total / count
        print(f"Average of '{column_name}': {avg}")

# Example usage
print(average_boolean_column("Levels_of_Accuracy.csv", "Any_Equality"))


Average of 'Any_Equality': 0.9740016760538676
None


In [3]:
import pandas as pd
def extract_fully_mismatched_rows(df):
    """
    Returns only rows where the final equality column (Any_Equality) is False.
    Does NOT modify AWS_Value or Manual_Value in any way.
    """

    col = "Any_Equality"

    # Work on a temporary boolean mask only
    mask = df[col].astype(str).str.upper() == "FALSE"

    return df[mask].copy()

df = pd.read_csv("Levels_of_Accuracy.csv")

bad_rows = extract_fully_mismatched_rows(df)

print(bad_rows.head())
bad_rows.to_csv("rows_with_no_equality.csv", index=False)


                                            file_name  row_index  col_index  \
191  padded_cdi_ca_1956_wk_prov_dbs_Part10_tables.csv          4         35   
272  padded_cdi_ca_1956_wk_prov_dbs_Part10_tables.csv          6         38   
350  padded_cdi_ca_1956_wk_prov_dbs_Part10_tables.csv          8         38   
529  padded_cdi_ca_1956_wk_prov_dbs_Part10_tables.csv         13         22   
584  padded_cdi_ca_1956_wk_prov_dbs_Part10_tables.csv         14         38   

    AWS_Value Manual_Value  Equal  Equal_No_Commas_No_Spaces  Equal_Shifted  \
191         5            6  False                      False          False   
272        61            7  False                      False          False   
350         1            9  False                      False          False   
529        33           38  False                      False          False   
584        77           15  False                      False          False   

     Only_Numbers_Equal_Or_Shifted Any_Equality Co

In [1]:
import pandas as pd
import nltk

def add_levenshtein(input_csv, output_csv=None):
    # Read as strings to avoid type weirdness
    df = pd.read_csv(input_csv, dtype=str, keep_default_na=False, na_filter=False)

    # Safety: make sure the expected columns exist
    if "AWS_Value" not in df.columns or "Manual_Value" not in df.columns:
        raise ValueError("Expected columns 'AWS_Value' and 'Manual_Value' not found.")

    # Compute edit distance row-wise
    df["Lev_Dist"] = [
        nltk.edit_distance(a, b)
        for a, b in zip(df["AWS_Value"], df["Manual_Value"])
    ]

    # Optional: normalized similarity [0,1]
    max_len = df["AWS_Value"].str.len().combine(df["Manual_Value"].str.len(), max)
    max_len = max_len.replace(0, 1)  # avoid division by zero
    df["Lev_Sim"] = 1 - df["Lev_Dist"] / max_len

    # Save result
    if output_csv is None:
        # default: overwrite input
        output_csv = input_csv

    df.to_csv(output_csv, index=False)
    print(f"Saved with Levenshtein: {output_csv}")

    return df
add_levenshtein("rows_with_no_equality.csv")

Saved with Levenshtein: rows_with_no_equality.csv


,file_name,row_index,col_index,AWS_Value,Manual_Value,Equal,Equal_No_Commas_No_Spaces,Equal_Shifted,Only_Numbers_Equal_Or_Shifted,Any_Equality,Confidence_Value,Lev_Dist,Lev_Sim
0,padded_cdi_ca_1956_wk_prov_dbs_Part10_tables.csv,4,35,5,6,False,False,False,False,False,87.26,1,0.000000
1,padded_cdi_ca_1956_wk_prov_dbs_Part10_tables.csv,6,38,61,7,False,False,False,False,False,85.6,2,0.000000
2,padded_cdi_ca_1956_wk_prov_dbs_Part10_tables.csv,8,38,1,9,False,False,False,False,False,82.37,1,0.000000
3,padded_cdi_ca_1956_wk_prov_dbs_Part10_tables.csv,13,22,33,38,False,False,False,False,False,51.86,1,0.500000
4,padded_cdi_ca_1956_wk_prov_dbs_Part10_tables.csv,14,38,77,15,False,False,False,False,False,85.94,2,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1329,padded_cdi_ca_1956_wk_prov_dbs_Part9_tables.csv,14,30,7,1,False,False,False,False,False,74.66,1,0.000000
1330,padded_cdi_ca_1956_wk_prov_dbs_Part9_tables.csv,14,38,74,15,False,False,False,False,False,79.93,2,0.000000
1331,padded_cdi_ca_1956_wk_prov_dbs_Part9_tables.csv,21,32,R.P. 50,50,False,False,False,False,False,74.02,5,0.285714
1332,padded_cdi_ca_1956_wk_prov_dbs_Part9_tables.csv,21,34,A.P. 47,47,False,False,False,False,False,47.58,5,0.285714


In [4]:
import pandas as pd
import re

def keep_only_numeric_decimal_space_comma_rows(input_csv, output_csv=None):
    df = pd.read_csv(input_csv, dtype=str, keep_default_na=False, na_filter=False)

    # Allowed: digits, dots, spaces, commas
    pattern = re.compile(r'^[0-9., ]+$')

    def valid(val):
        return bool(pattern.fullmatch(val.strip()))

    # Keep rows where BOTH columns match
    mask = df["AWS_Value"].apply(valid) & df["Manual_Value"].apply(valid)

    filtered = df[mask].copy()

    if output_csv:
        filtered.to_csv(output_csv, index=False)
        print(f"Saved filtered file: {output_csv}")

    return filtered

keep_only_numeric_decimal_space_comma_rows("rows_with_no_equality.csv", "rows_numeric_decimal_space_comma.csv")

Saved filtered file: rows_numeric_decimal_space_comma.csv


,file_name,row_index,col_index,AWS_Value,Manual_Value,Equal,Equal_No_Commas_No_Spaces,Equal_Shifted,Only_Numbers_Equal_Or_Shifted,Any_Equality,Confidence_Value,Lev_Dist,Lev_Sim
0,padded_cdi_ca_1956_wk_prov_dbs_Part10_tables.csv,4,35,5,6,False,False,False,False,False,87.26,1,0.0
1,padded_cdi_ca_1956_wk_prov_dbs_Part10_tables.csv,6,38,61,7,False,False,False,False,False,85.6,2,0.0
2,padded_cdi_ca_1956_wk_prov_dbs_Part10_tables.csv,8,38,1,9,False,False,False,False,False,82.37,1,0.0
3,padded_cdi_ca_1956_wk_prov_dbs_Part10_tables.csv,13,22,33,38,False,False,False,False,False,51.86,1,0.5
4,padded_cdi_ca_1956_wk_prov_dbs_Part10_tables.csv,14,38,77,15,False,False,False,False,False,85.94,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1326,padded_cdi_ca_1956_wk_prov_dbs_Part9_tables.csv,9,34,.... 48,48,False,False,False,False,False,40.28,5,0.2857142857142857
1327,padded_cdi_ca_1956_wk_prov_dbs_Part9_tables.csv,9,38,9 10,10,False,False,False,False,False,72.51,2,0.5
1328,padded_cdi_ca_1956_wk_prov_dbs_Part9_tables.csv,12,4,7,1,False,False,False,False,False,86.82,1,0.0
1329,padded_cdi_ca_1956_wk_prov_dbs_Part9_tables.csv,14,30,7,1,False,False,False,False,False,74.66,1,0.0
